# 07 — Explainability and Error Analysis
Inspect where the selected models fail and which features most influence predictive performance.

In [ ]:
import pandas as pd
from src.features import prepare_feature_frame, build_preprocessor
from src.modeling import grouped_train_test_split, derive_high_cost_labels, get_regression_candidates, get_classification_candidates
from src.evaluation import regression_error_table, classification_error_table, permutation_importance_table
df = pd.read_csv('data/processed/analytical_dataset.csv')
X, y, groups = prepare_feature_frame(df)
Xtr, Xte, ytr, yte, gtr, gte = grouped_train_test_split(X, y, groups)
preprocessor = build_preprocessor(Xtr)

In [ ]:
reg = get_regression_candidates(preprocessor)['Random Forest'].fit(Xtr, ytr)
rpred = reg.predict(Xte)
regression_error_table(pd.DataFrame({'Customer ID': gte.values}), yte.reset_index(drop=True), rpred).head(10)

In [ ]:
permutation_importance_table(reg, Xte, yte, 'neg_mean_absolute_error', n_repeats=5).head(10)

In [ ]:
ytr_c, yte_c, threshold = derive_high_cost_labels(ytr, yte)
clf = get_classification_candidates(preprocessor)['HistGradientBoosting'].fit(Xtr, ytr_c)
score = clf.predict_proba(Xte)[:, 1]
pred = (score >= 0.5).astype(int)
classification_error_table(pd.DataFrame({'Customer ID': gte.values}), yte_c.reset_index(drop=True), pred, score).query("error_type != 'correct'").head(10)

In [ ]:
permutation_importance_table(clf, Xte, yte_c, 'average_precision', n_repeats=5).head(10)

Permutation importance describes model dependence on features for predictive performance. It does not establish medical causality.